The code below is designed to crawl keywords from the specified pages and save the data into a CSV file. If access to a link is denied, it will continuously attempt to access the link until successful or until it reaches the maximum number of retries that we have set. To avoid being blocked by the website, we use different user agents to mimic a normal user rather than a bot. Additionally, we set cookies to ensure access to the website with our desired settings, specifically to set the delivery destination to New York.

In [42]:
import requests
from bs4 import BeautifulSoup
import cloudscraper
import time
import random
import csv
import pandas as pd
import re



def convert_sales(sales_str):
    match = re.search(r'(\d+)(k)?', sales_str, re.IGNORECASE)
    if match:
        number = int(match.group(1))
        if match.group(2):
            return number 
        else:
            return number / 1000 
    return 0 
        



def get_product_cards(url, i, max_retries=6):
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Gecko/20100101 Firefox/89.0',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36 Edge/18.19041',
    ]

    headers = {
        'User-Agent': random.choice(user_agents),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Referer': 'https://www.google.com/',  
        'DNT': '1'
        }
    
    cookies = {
        'session-id': '144-3606130-3973724',
        'ubid-main': '133-9253987-1457448',
        "x-main": "xm@BSjhmV0gdhmbh06xQ4jUMESRC9LJ2aw7yjxDvG3vzvrp1IYs0hgLRXKKnoXmN"
        }
    product_data_list = {i: []}

    retry_count = 0
    while retry_count < max_retries:
        try:
            scraper = cloudscraper.create_scraper()
            response = scraper.get(url, headers=headers, cookies=cookies)
            response.raise_for_status()  # This will raise an error for 4xx and 5xx responses

            # If we reach here, the request was successful
            soup = BeautifulSoup(response.text, 'html.parser')
            product_cards = soup.find_all(class_='s-result-item', attrs={"data-component-type": "s-search-result"})    

            for card in product_cards:
                title_element = card.find('span', class_='a-size-base-plus a-color-base a-text-normal')
                title_text = title_element.text.strip() if title_element else "No title found"

                sales_element = card.find('span', class_='a-size-base a-color-secondary')
                sales_text = sales_element.text.strip() if sales_element else "No sales info found"
                sales = convert_sales(sales_text)

                price_info_element = card.find('span', class_='a-price a-text-price')
                if price_info_element:
                    price_text_element = price_info_element.find_next('span', attrs={'aria-hidden': 'true'})
                    if price_text_element:
                        price_text = price_text_element.text.strip()
                    else:
                        price_text = "No price found"
                else:
                    price_text = "No price found"

                star = card.find("span", class_="a-icon-alt")
                star_text = star.text.strip() if star else "No star info found"

                comment_count = card.find("span", class_="a-size-base s-underline-text", attrs={"aria-hidden": "true"})
                comment_count_text = comment_count.text.strip() if comment_count else "No comment count info found"

                print(f'Title: {title_text}')
                print(f'Sales: {sales}')
                print(f'Price per count: {price_text}')
                print(f'Star: {star_text}')
                print(f'Comment count: {comment_count_text}')
                print('-' * 40) 

                product_data = {
                    "Sales": sales,
                    "Price per count": float(price_text.replace('$', '').strip()) if price_text not in ["No price found", ""] else 0.0,
                    "star": float(star_text.split(' ')[0]) if star_text.split(' ')[0].replace('.', '', 1).isdigit() else 0.0,
                    "comment count": int(comment_count_text.replace(',', '').strip()) if "No comment count info found" not in comment_count_text else 0
                }

                product_data_list[i].append(product_data)

            break  # Exit the retry loop if successful

        except requests.exceptions.HTTPError as e:
            if response.status_code == 503:
                print(f"503 error occurred. Retrying... ({retry_count + 1}/{max_retries})")
                retry_count += 1
            else:
                print(f"HTTP error occurred: {e}")
                break  # Exit the loop for errors other than 503
        except Exception as e:
            print(f"Failed to retrieve product cards: {e}")
            break  # Exit the loop on other exceptions

    return pd.DataFrame(product_data_list[i])

For coffee pods

The code below will loop multiple times based on the number of pages available for the requested products on the website, with each execution of the function generating a new DataFrame.
Expected output is the dataframe contains:'Price', 'Sales', 'Star', 'Comment'

In [43]:
t = []
for i in range(1, 8):
    url = f"https://www.amazon.com/s?k=coffee+pod&page={i}"
    print(f"Fetching data from: {url}")  
    p_df = get_product_cards(url, i)

    print(f"DataFrame for page {i}:\n{p_df}") 

    if not p_df.empty:
        t.append(p_df) 
    else:
        print(f"Received an empty DataFrame for page {i}.")

    time.sleep(random.uniform(1, 3)) 

if t:
    coffee_pod_data = pd.concat(t, ignore_index=True) 
    print("final result")
    print(coffee_pod_data) 
else:
    print("No valid data retrieved.")

Fetching data from: https://www.amazon.com/s?k=coffee+pod&page=1
Title: Amazon Brand - Happy Belly Medium Roast Coffee Pods, Donut Style, Compatible with Keurig 2.0 K-Cup Brewers, 100 Count
Sales: 40
Price per count: $0.24
Star: 4.3 out of 5 stars
Comment count: 65,673
----------------------------------------
Title: Black Rifle Coffee Company Supply Drop Variety Pack (96 Count of K Cups) Contains a Mix of Silencer Smooth (Light Roast), AK-47 (Medium Roast), Just Black (Medium Roast), and Beyond Black (Dark Roast)
Sales: 3
Price per count: $0.86
Star: 4.6 out of 5 stars
Comment count: 7,668
----------------------------------------
Title: San Francisco Bay Compostable Coffee Pods - Original Variety Pack (80 Ct) K Cup Compatible including Keurig 2.0, French, Breakfast, Fog, Organic Rainforest
Sales: 20
Price per count: $0.55
Star: 4.4 out of 5 stars
Comment count: 91,724
----------------------------------------
Title: Victor Allen's Coffee Variety Pack (Morning Blend, Donut Shop Blend, an

Next, we clean the data by first filtering out the outliers. We then invert the star rating using the formula '5 - star', which allows us to reverse the meaning of the x-axis so that better values are on the left and worse values are on the right. Finally, we remove any entries that contain a zero in any feature, as this indicates missing values.
Finally, it will generate a csv.

In [ ]:
Q1 = coffee_pod_data['Price per count'].quantile(0.25)
Q3 = coffee_pod_data['Price per count'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
coffee_pod_data_1 = coffee_pod_data[(coffee_pod_data['Price per count'] >= lower_bound) & (coffee_pod_data['Price per count'] <= upper_bound)]
coffee_pod_data['star'] = 5 - coffee_pod_data['star']
coffee_pod_data_0 = coffee_pod_data_1[(coffee_pod_data_1['Sales'] > 0) & (coffee_pod_data_1['Price per count'] > 0)]
coffee_pod_data_0.to_csv('coffee_pod_data.csv', index=False, sep=',', encoding='utf-8')

Sanitary Pad

In [45]:
y = []
for i in range(1, 27):
    url = f"https://www.amazon.com/s?k=sanitary+pads&i=hpc&rh=n%3A3760901%2Cp_n_feature_five_browse-bin%3A116422830011&dc&page={i}"
    print(f"Fetching data from: {url}")  
    s_df = get_product_cards(url, i)

    print(f"DataFrame for page {i}:\n{p_df}") 

    if not s_df.empty:
        y.append(s_df) 
    else:
        print(f"Received an empty DataFrame for page {i}.")

    time.sleep(random.uniform(1, 3)) 

if y:
    sanitary_pad = pd.concat(y, ignore_index=True) 
    print("final result")
    print(sanitary_pad) 
else:
    print("No valid data retrieved.")

Fetching data from: https://www.amazon.com/s?k=sanitary+pads&i=hpc&rh=n%3A3760901%2Cp_n_feature_five_browse-bin%3A116422830011&dc&page=1
Title: Comfort Pads (Regular, 32 Count) - Ultra Soft, Comfortable, Thin Pads with Wings, Super Absorbency. Premium Quality
Sales: 2
Price per count: $0.28
Star: 4.4 out of 5 stars
Comment count: 305
----------------------------------------
Title: U by Kotex Clean & Secure Ultra Thin Pads with Wings, Regular Absorbency, 216 Count (6 Packs of 36) (Packaging May Vary)
Sales: 1
Price per count: $0.16
Star: 4.6 out of 5 stars
Comment count: 434
----------------------------------------
Title: U by Kotex Clean & Secure Maxi Pads, Regular Absorbency, 192 Count (4 Packs of 48) (Packaging May Vary)
Sales: 3
Price per count: $0.15
Star: 4.5 out of 5 stars
Comment count: 3,029
----------------------------------------
Title: Amazon Basics Thick Maxi Pads for Periods, Super Absorbency, Unscented, 48 Count, 1 Pack (Previously Solimo)
Sales: 50
Price per count: $0.11

In [46]:
Q1 = sanitary_pad['Price per count'].quantile(0.25)
Q3 = sanitary_pad['Price per count'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
sanitary_pad['star'] = 5 - sanitary_pad['star']
sanitary_pad_1 = sanitary_pad[(sanitary_pad['Price per count'] >= lower_bound) & (sanitary_pad['Price per count'] <= 2)]
sanitary_pad_0 = sanitary_pad_1[(sanitary_pad_1['Sales'] > 0) & (sanitary_pad_1['Price per count'] > 0)]
sanitary_pad_0.to_csv('sanitary_pad_data.csv', index=False, sep=',', encoding='utf-8')



Induction cooktop

This function is essentially the same as the previous get_product_cards function, but it updates the class names of certain keywords to match the new structure of each product card.

In [50]:
def convert_sales(sales_str):
    match = re.search(r'(\d+)(k)?', sales_str, re.IGNORECASE)
    if match:
        number = int(match.group(1))
        if match.group(2):
            return number 
        else:
            return number / 1000 
    return 0 
        



def i_get_product_cards(url, i, max_retries=5):
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Gecko/20100101 Firefox/89.0',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36 Edge/18.19041',
    ]

    headers = {
        'User-Agent': random.choice(user_agents),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Referer': 'https://www.google.com/',  
        'DNT': '1'
        }
    
    cookies = {
        'session-id': '144-3606130-3973724',
        'ubid-main': '133-9253987-1457448',
        "x-main": "xm@BSjhmV0gdhmbh06xQ4jUMESRC9LJ2aw7yjxDvG3vzvrp1IYs0hgLRXKKnoXmN"
        }
    
    product_data_list = {i: []}
    retry_count = 0
    while retry_count < max_retries:
        try:
            scraper = cloudscraper.create_scraper()
            response = scraper.get(url, headers=headers, cookies=cookies)
            response.raise_for_status() 
            soup = BeautifulSoup(response.text, 'html.parser')

            product_cards = soup.find_all(class_='s-result-item', attrs={"data-component-type": "s-search-result"})    
            for card in product_cards:
                
                title_element = card.find('span', class_='a-size-base-plus a-color-base a-text-normal')
                title_text = title_element.text.strip() if title_element else "No title found"
                

                sales_element = card.find('span', class_='a-size-base a-color-secondary')
                sales_text = sales_element.text.strip() if sales_element else "No sales info found"
                sales = convert_sales(sales_text)

                price_info_element = card.find('span', class_='a-offscreen')
                if price_info_element:
                    price_text = price_info_element.text.strip()
                else:
                    price_text = "No price found"
            
                    
                star = card.find("span", class_="a-icon-alt")
                star_text = star.text.strip() if star else "No start info found"
                
                comment_count = card.find("span", class_="a-size-base s-underline-text", attrs={"aria-hidden": "true"})
                comment_count_text = comment_count.text.strip() if comment_count else "No comment count info found"

                print(f'Title: {title_text}')
                print(f'Sales: {sales}')
                print(f'Price per count: {price_text}')
                print(f'star: {star_text}')
                print(f'comment count: {comment_count_text}')
                print('-' * 40) 

                product_data = {
                    "Sales": sales,
                    "Price per count": float(price_text.replace('$', '').strip()) if price_text not in ["No price found", ""] else 0.0,
                    "star": float(star_text.split(' ')[0]) if star_text.split(' ')[0].replace('.', '', 1).isdigit() else 0.0,
                    "comment count": int(comment_count_text.replace(',', '').strip()) if "No comment count info found" not in comment_count_text else 0
                }
                
                product_data_list[i].append(product_data)
        
                
            break  # Exit the retry loop if successful

        except requests.exceptions.HTTPError as e:
            if response.status_code == 503:
                print(f"503 error occurred. Retrying... ({retry_count + 1}/{max_retries})")
                retry_count += 1
            else:
                print(f"HTTP error occurred: {e}")
                break  # Exit the loop for errors other than 503
        except Exception as e:
            print(f"Failed to retrieve product cards: {e}")
            break  # Exit the loop on other exceptions

    return pd.DataFrame(product_data_list[i])

In [52]:
u = []
for i in range(1, 21):
    url = f"https://www.amazon.com/s?k=induction+cooktop&page={i}"
    print(f"Fetching data from: {url}")  
    i_df = i_get_product_cards(url, i)

    print(f"DataFrame for page {i}:\n{i_df}") 

    if not i_df.empty:
        u.append(i_df) 
    else:
        print(f"Received an empty DataFrame for page {i}.")

    time.sleep(random.uniform(1, 3)) 

if t:
    induction_cooktop = pd.concat(u, ignore_index=True) 
    print("final result")
    print(induction_cooktop) 
else:
    print("No valid data retrieved.")

Fetching data from: https://www.amazon.com/s?k=induction+cooktop&page=1
503 error occurred. Retrying... (1/5)
503 error occurred. Retrying... (2/5)
503 error occurred. Retrying... (3/5)
Title: No title found
Sales: 0.8
Price per count: $58.49
star: 4.2 out of 5 stars
comment count: 1,294
----------------------------------------
Title: No title found
Sales: 0.8
Price per count: $125.99
star: 4.5 out of 5 stars
comment count: 885
----------------------------------------
Title: No title found
Sales: 2
Price per count: $69.99
star: 4.6 out of 5 stars
comment count: 403
----------------------------------------
Title: No title found
Sales: 3
Price per count: $116.96
star: 4.4 out of 5 stars
comment count: 7,912
----------------------------------------
Title: No title found
Sales: 2
Price per count: $59.99
star: 4.5 out of 5 stars
comment count: 7,053
----------------------------------------
Title: No title found
Sales: 0.7
Price per count: $160.99
star: 4.6 out of 5 stars
comment count: 273


In [53]:
Q1 = induction_cooktop['Price per count'].quantile(0.25)
Q3 = induction_cooktop['Price per count'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
induction_cooktop['star'] = 5 - induction_cooktop['star']
induction_cooktop_1 = induction_cooktop[(induction_cooktop['Price per count'] >= lower_bound) & (induction_cooktop['Price per count'] <= upper_bound)]
induction_cooktop_0 = induction_cooktop_1[(induction_cooktop_1['Sales'] > 0) & (induction_cooktop_1['Price per count'] > 0)]
induction_cooktop_0.to_csv('induction_cooktop_data.csv', index=False, sep=',', encoding='utf-8')

The below code is to make a new dataframe that combine all product data and out put a csv.

In [54]:
all_data = pd.concat([coffee_pod_data_0, sanitary_pad_0, induction_cooktop_0])
all_data.to_csv('cross_data.csv', index=False, sep=',', encoding='utf-8')